# Chapter 7
## Linear Integrate and Fire (LIF) Neurons
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### The Hodgkin-Huxley Neuron's Subthreshold Response

Underlies Figures 7.1, 7.2 and 7.3: a subthreshold current step
(`i_ext=7`, below the HH neuron's firing threshold when starting from
v=-20mV) drives the same relaxation used to motivate the LIF
approximation, extract the effective membrane time constant, and show
the individual ionic currents.

In [ ]:
def simulate_HH_subthreshold(simulation_time):
    """The book's chapter-1 HH neuron, started at v=-20mV (subthreshold)."""

    El = -59 * b2.mV
    EK = -82 * b2.mV
    ENa = 45 * b2.mV
    gl = 0.3 * b2.msiemens
    gK = 36 * b2.msiemens
    gNa = 120 * b2.msiemens
    C = 1 * b2.ufarad

    eqs = """
    I_e : amp
    membrane_Im = I_e + gNa*m**3*h*(ENa-vm) + \
        gl*(El-vm) + gK*n**4*(EK-vm) : amp

    alphan = 0.01/mV * (-60.0*mV - vm) / (exp((-60.0*mV - vm) / (10.0*mV)) - 1.0)/ms: Hz
    alpham = (vm + 45.0*mV) / (10.0*mV) / (1.0 - exp(-(vm + 45.0*mV) / (10.0*mV)))/ms : Hz
    alphah = 0.07*exp(-(vm + 70*mV)/(20*mV))/ms : Hz

    betan = 0.125 * exp(-(vm + 70.0*mV) / (80.0*mV))/ms: Hz
    betam = 4.0 * exp(-(vm + 70.0*mV) / (18.0*mV))/ms: Hz
    betah = 1. / (exp(-(vm + 40.0*mV) / (10.0*mV)) + 1.0)/ms : Hz

    dn/dt = alphan*(1-n)-betan*n : 1
    dm/dt = alpham*(1-m)-betam*m : 1
    dh/dt = alphah*(1-h)-betah*h : 1

    dvm/dt = membrane_Im/C : volt
    """

    b2.defaultclock.dt = 0.01 * b2.ms
    neuron = b2.NeuronGroup(1, eqs, method="exponential_euler")
    neuron.vm = -20 * b2.mV
    neuron.I_e = 7 * b2.uA

    # initial conditions at their steady state for v=-20mV
    neuron.m = 0.9163
    neuron.h = 0.00648
    neuron.n = 0.8590

    st_mon = b2.StateMonitor(neuron, ["vm", "m", "n", "h"], record=True)
    net = b2.Network(neuron)
    net.add(st_mon)
    net.run(simulation_time)

    return st_mon


sm = simulate_HH_subthreshold(100 * b2.ms)

### Figure 7.1
LIF Fit to the HH Neuron's Subthreshold Response

In [ ]:
t = sm.t / b2.ms
v = sm.vm[0] / b2.mV

fig, ax = plt.subplots(figsize=(7, 3))
v_line = np.asarray([-82.0, -54.0])
t_line = np.asarray([0.0, (v_line[1] - v_line[0]) / 1.65])
for i in range(6):
    ax.plot(t_line + t_line[1] * i, v_line, c="b", lw=2)
    ax.plot([t_line[1] * i] * 2, v_line, c="b", lw=2, ls="--")

ax.plot(t, v, lw=2, c="k")
ax.set_xlim(0, np.max(t))
ax.set_ylim(-100, 50)
ax.set_xlabel("t [ms]")
ax.set_ylabel("v [mV]")
plt.tight_layout()
plt.show()

### Figure 7.2
Effective Membrane Time Constant of the HH Neuron

In [ ]:
gl = 0.3
gK = 36
gNa = 120
m = sm.m[0]
n = sm.n[0]
h = sm.h[0]
tau = 1 / (gK * n**4 + gNa * m**3 * h + gl)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, tau, lw=2, c="k")
ax.set_xlim(0, np.max(t))
ax.set_xlabel("t [ms]")
ax.set_ylabel(r"$\tau$ [ms]")
plt.tight_layout()
plt.show()

### Figure 7.3
The Individual Ionic Currents Underlying the Subthreshold Response

In [ ]:
El, EK, ENa = -59, -82, 45
I_na = gNa * m**3 * h * (ENa - v)
I_k = gK * n**4 * (EK - v)
I_l = gl * (El - v)
I_tot = I_na + I_k + I_l

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, I_na, lw=2, c="r", label=r"$I_{na}$")
ax.plot(t, I_k, lw=2, c="g", label=r"$I_{k}$")
ax.plot(t, I_l, lw=2, c="b", label=r"$I_{l}$")
ax.plot(t, I_tot, lw=2, c="k", label=r"$I_{tot}$")
ax.legend()
ax.set_xlim(0, np.max(t))
ax.set_ylim(-20, 20)
ax.set_xlabel("t [ms]")
ax.set_ylabel(r"$I\ [\mu A/cm^2]$")
plt.tight_layout()
plt.show()

### The Linear Integrate-and-Fire (LIF) Neuron

Dimensionless model (threshold at v=1, reset to v=0), same convention
as the book's MATLAB scripts and the Python port -- v, tau_m and I have
no physical units here, and "t" is in the same arbitrary time unit as
tau_m.

In [ ]:
def simulate_LIF_neuron(tau_m, I, simulation_time, dt=0.01 * b2.ms):
    eqs = "dv/dt = (-v/tau_m + I)/ms : 1"
    neuron = b2.NeuronGroup(1, eqs, threshold="v>1", reset="v=0",
                             method="rk4", dt=dt,
                             namespace={"tau_m": tau_m, "I": I})
    neuron.v = 0
    st_mon = b2.StateMonitor(neuron, "v", record=True)
    sp_mon = b2.SpikeMonitor(neuron)
    net = b2.Network(neuron)
    net.add(st_mon, sp_mon)
    net.run(simulation_time)
    return st_mon, sp_mon

### Figure 7.4
LIF Voltage Trace

In [ ]:
sm4, spm4 = simulate_LIF_neuron(tau_m=10, I=0.11, simulation_time=100 * b2.ms)
t4 = sm4.t / b2.ms
v4 = sm4.v[0]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t4, v4, lw=2, c="k")
ax.set_xlim(0, np.max(t4))
ax.set_ylim(0, 2)
ax.set_xlabel("t")
ax.set_ylabel("v")
plt.tight_layout()
plt.show()

### Figure 7.5
LIF Voltage Trace (tau_m=2, current tuned for a 20-time-unit period)

In [ ]:
tau_m2 = 2
I2 = 1 / (1 - np.exp(-20.0 / tau_m2)) / tau_m2
sm5, spm5 = simulate_LIF_neuron(tau_m=tau_m2, I=I2, simulation_time=50 * b2.ms)
t5 = sm5.t / b2.ms
v5 = sm5.v[0]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t5, v5, lw=2, c="k")
ax.set_xlim(0, np.max(t5))
ax.set_ylim(0, 2)
ax.set_xlabel("t")
ax.set_ylabel("v")
plt.tight_layout()
plt.show()